<a href="https://colab.research.google.com/github/tsarangler/ECON3916-Statistical-Machine-Learning/blob/main/lab%2012/Lab_12_Architecting_the_Prediction_Engine_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/tsarangler/ECON3916-Statistical-Machine-Learning/main/data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

In [7]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [8]:
import statsmodels.formula.api as smf
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula,data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 13 Apr 2026   Prob (F-statistic):          2.81e-309
Time:                        18:43:30   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [9]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [10]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df["Home_Value"],y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [12]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 0 ── Reproducible example (swap in your own `results` object)
# ─────────────────────────────────────────────────────────────────────────────
np.random.seed(42)
n = 300
X_raw = np.random.rand(n, 3) * 100          # 3 synthetic features
true_beta = np.array([2.5, -1.2, 0.8])
y = X_raw @ true_beta + np.random.normal(0, 15, n)

# statsmodels requires an explicit intercept column
X = sm.add_constant(X_raw)                  # shape: (n, 4)

# Fit OLS — `results` is the object your lab already produced
results = sm.OLS(y, X).fit()

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1 ── Extract diagnostics from the statsmodels results object
# ─────────────────────────────────────────────────────────────────────────────

# results.fittedvalues  →  ŷ  (in-sample predictions, length n)
fitted = pd.Series(results.fittedvalues)          # pd.Series aligned to original index

# results.resid  →  ε̂ = y − ŷ  (raw OLS residuals, NOT standardised yet)
residuals = pd.Series(results.resid)              # pd.Series; same index as fittedvalues

# Compute the residual standard deviation (ddof=0 because statsmodels
# already accounts for degrees of freedom internally; here we just want
# a scalar threshold for visual flagging, so population std is fine)
resid_std = residuals.std(ddof=0)

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 2 ── Classify each observation as normal or outlier
# ─────────────────────────────────────────────────────────────────────────────

# Boolean mask: True when the absolute residual exceeds 2 σ
outlier_mask = residuals.abs() > 2 * resid_std

# Split into two sub-series for separate trace styling
fitted_normal   = fitted[~outlier_mask]
residual_normal = residuals[~outlier_mask]

fitted_outlier   = fitted[outlier_mask]
residual_outlier = residuals[outlier_mask]

# Build a hover label that shows the observation index + both values
def hover_text(f_series, r_series, label):
    return [
        f"<b>{label}</b><br>"
        f"Obs index: {i}<br>"
        f"Fitted ŷ: {fv:.2f}<br>"
        f"Residual ε̂: {rv:.2f}"
        for i, fv, rv in zip(f_series.index, f_series, r_series)
    ]

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3 ── Build the Plotly figure using graph_objects (no deprecated calls)
# ─────────────────────────────────────────────────────────────────────────────

fig = go.Figure()

# ── Trace 1: normal residuals ────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=fitted_normal,
    y=residual_normal,
    mode="markers",
    name="Normal residual (|ε̂| ≤ 2σ)",
    marker=dict(
        color="steelblue",
        size=7,
        opacity=0.65,
        line=dict(width=0.5, color="white")
    ),
    hovertemplate="%{text}<extra></extra>",   # suppress default trace label
    text=hover_text(fitted_normal, residual_normal, "Normal")
))

# ── Trace 2: outlier residuals ───────────────────────────────────────────────
# Stark crimson (#DC143C) makes structural deviations immediately legible
fig.add_trace(go.Scatter(
    x=fitted_outlier,
    y=residual_outlier,
    mode="markers",
    name="Outlier (|ε̂| > 2σ)",
    marker=dict(
        color="#DC143C",      # crimson — WCAG-compliant contrast on dark bg
        size=10,
        opacity=0.9,
        symbol="diamond",
        line=dict(width=1, color="white")
    ),
    hovertemplate="%{text}<extra></extra>",
    text=hover_text(fitted_outlier, residual_outlier, "⚠ Outlier")
))

# ── Reference lines ──────────────────────────────────────────────────────────
x_min, x_max = fitted.min(), fitted.max()

# Zero line: ideal residual mean should hover here (OLS guarantee)
fig.add_shape(
    type="line", x0=x_min, x1=x_max, y0=0, y1=0,
    line=dict(color="white", width=1.5, dash="solid")
)

# ±2σ envelope lines — visual band for the outlier threshold
for sign, label in [(1, "+2σ threshold"), (-1, "−2σ threshold")]:
    fig.add_shape(
        type="line",
        x0=x_min, x1=x_max,
        y0=sign * 2 * resid_std, y1=sign * 2 * resid_std,
        line=dict(color="#FFD700", width=1.2, dash="dot")
    )
    # Annotate only the upper band to avoid clutter
    if sign == 1:
        fig.add_annotation(
            x=x_max, y=sign * 2 * resid_std,
            text=f" {label} ({sign * 2 * resid_std:.1f})",
            showarrow=False, font=dict(color="#FFD700", size=11),
            xanchor="right"
        )

# ── Layout ───────────────────────────────────────────────────────────────────
n_outliers = outlier_mask.sum()
pct_outliers = 100 * n_outliers / len(residuals)

fig.update_layout(
    title=dict(
        text=(
            f"<b>Residual Forensics Dashboard</b>  "
            f"<span style='font-size:13px;color:gray'>"
            f"n={len(residuals)} obs · "
            f"{n_outliers} outliers ({pct_outliers:.1f}%) · "
            f"RMSE={np.sqrt(results.mse_resid):.3f} · "
            f"R²={results.rsquared:.4f}"
            f"</span>"
        ),
        font=dict(size=17)
    ),
    xaxis=dict(
        title="Fitted Values (ŷ)",
        showgrid=True, gridcolor="rgba(255,255,255,0.08)"
    ),
    yaxis=dict(
        title="Residuals (ε̂ = y − ŷ)",
        showgrid=True, gridcolor="rgba(255,255,255,0.08)",
        zeroline=False   # we drew our own zero line above for style control
    ),
    plot_bgcolor="#0f1117",
    paper_bgcolor="#0f1117",
    font=dict(color="white", family="monospace"),
    legend=dict(
        bgcolor="rgba(255,255,255,0.05)",
        bordercolor="rgba(255,255,255,0.15)",
        borderwidth=1
    ),
    hovermode="closest",
    margin=dict(l=60, r=40, t=80, b=60)
)

fig.show()

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 4 ── Print a concise outlier report to stdout
# ─────────────────────────────────────────────────────────────────────────────
outlier_report = pd.DataFrame({
    "fitted_y"  : fitted_outlier.values,
    "residual"  : residual_outlier.values,
    "std_devs"  : (residual_outlier / resid_std).values   # signed σ-distance
}, index=fitted_outlier.index).sort_values("std_devs", key=abs, ascending=False)

print(f"\n{'─'*55}")
print(f" TOP OUTLIERS  (|ε̂| > 2σ,  σ = {resid_std:.3f})")
print(f"{'─'*55}")
print(outlier_report.round(3).to_string())
print(f"{'─'*55}\n")


───────────────────────────────────────────────────────
 TOP OUTLIERS  (|ε̂| > 2σ,  σ = 14.349)
───────────────────────────────────────────────────────
     fitted_y  residual  std_devs
281   258.284   -39.044    -2.721
59     -4.039    38.803     2.704
184   142.079    36.621     2.552
66    181.957    35.993     2.508
237   126.693   -34.407    -2.398
151   255.159    34.326     2.392
126   181.420    32.311     2.252
213   118.380    31.747     2.213
12    119.630   -31.688    -2.208
295   -14.649   -30.616    -2.134
173    -3.319    30.252     2.108
279    72.139   -28.973    -2.019
───────────────────────────────────────────────────────

